<a href="https://colab.research.google.com/github/bernirazquin/Fisheries_data/blob/main/gfw_data_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Global Fishing Watch raw data download

This script will download raw data from the global fishing watch dataset in BigQuery.

It allows to choose date, region and type of data.

We will use it to download the fishing efforts through 2020 - 2024 on Drafting longline vessels, to overlay shark species of interest distribution.

This project is made alongside SharkProject International.

In [2]:
from google.colab import auth
from google.cloud import bigquery
from google.colab import files
import pandas as pd

print("Authenticating user...")
auth.authenticate_user()

project_id = 'sharkproject-int'
client = bigquery.Client(project=project_id)

# Execute query on GFW v3 database using the exact schema
query = """
SELECT
  cell_ll_lat AS lat_grid,
  cell_ll_lon AS lon_grid,
  geartype,
  CASE
    WHEN geartype IN ('drifting_longlines', 'tuna_purse_seines', 'purse_seines') THEN 'Pelagic'
    WHEN geartype IN ('trawlers', 'set_longlines', 'set_gillnets', 'dredge_fishing') THEN 'Benthic'
    ELSE 'Other'
  END AS habitat_zone,
  SUM(fishing_hours) AS total_fishing_hours
FROM
  `global-fishing-watch.fishing_effort_v3.fleet_monthly_10_v3`
WHERE
  -- filtered the dedicated 'year' column
  year BETWEEN 2020 AND 2024

  -- Gear filter
  AND geartype IN (
      'drifting_longlines',
      'tuna_purse_seines',
      'purse_seines',
      'trawlers',
      'set_longlines',
      'set_gillnets',
      'dredge_fishing'
  )

  -- Spatial filter (NE Atlantic & Mediterranean)
  AND cell_ll_lat BETWEEN 25.0 AND 60.0
  AND cell_ll_lon BETWEEN -35.0 AND 30.0

  -- Exclude zero effort
  AND fishing_hours > 0
GROUP BY
  lat_grid,
  lon_grid,
  geartype,
  habitat_zone
ORDER BY
  total_fishing_hours DESC
"""

print("Executing BigQuery, it can take a moment...")

# Run the query and convert to DataFrame
df = client.query(query).to_dataframe()
print(f"Query successful. Extracted {len(df)} grid cells.")

# Export and download
output_filename = 'GFW_Master_Effort_Europe_2020_2024.csv'
df.to_csv(output_filename, index=False)
print(f"Saved to '{output_filename}'. Initiating download...")

# Download to your local machine
files.download(output_filename)

Authenticating user...
Executing BigQuery, it can take a moment...
Query successful. Extracted 150626 grid cells.
Saved to 'GFW_Master_Effort_Europe_2020_2024.csv'. Initiating download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>